In [1]:
!pip install -q datasets evaluate seqeval

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.6 MB/s eta 0:00:00


In [2]:
from datasets import load_dataset

ds = load_dataset("thainq107/abte-restaurants")
# Cùng dataset với ATSC, nhưng lần này ta dùng cột 'Tags' (BIO labels)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/454 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/183k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/61.8k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/3602 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1119 [00:00<?, ? examples/s]

In [3]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert/distilbert-base-uncased")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [6]:
# BIO labels: O=0, B-Term=1, I-Term=2
id2label = {0: "O", 1: "B-Term", 2: "I-Term"}
label2id = {"O": 0, "B-Term": 1, "I-Term": 2}

def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(examples["Tokens"], truncation=True, is_split_into_words=True)

    labels = []
    for i, label in enumerate(examples["Tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            # Special tokens have a word id that is None. We set the label to -100 so they are
            # ignored in the loss function.
            if word_idx is None:
                label_ids.append(-100)
            # We set the label for the first token of each word.
            elif word_idx != previous_word_idx:
                label_ids.append(int(label[word_idx]))
            # For the other tokens in a word, we set the label to the current B- or I- tag.
            # However, if it's an I- tag, we propagate it. If it's a B- tag, we also propagate it to ensure continuity.
            # A common strategy is to only label the first subword token and set others to -100.
            # For this task, we will align all subwords to the same tag as the first subword of the original word.
            else:
                # This is a subword token, assign it the same label as the first subword of the word.
                label_ids.append(int(label[word_idx]))
            previous_word_idx = word_idx

        labels.append(label_ids)
    tokenized_inputs["labels"] = labels
    return tokenized_inputs

preprocessed_ds = ds.map(tokenize_and_align_labels, batched=True)

Map:   0%|          | 0/3602 [00:00<?, ? examples/s]

Map:   0%|          | 0/1119 [00:00<?, ? examples/s]

In [7]:
from transformers import AutoModelForTokenClassification

# BIO labels: O=0, B-Term=1, I-Term=2
# id2label = {0: "O", 1: "B-Term", 2: "I-Term"}
# label2id = {"O": 0, "B-Term": 1, "I-Term": 2}

model = AutoModelForTokenClassification.from_pretrained(
    "distilbert/distilbert-base-uncased",
    num_labels=3,
    id2label=id2label,
    label2id=label2id
)
# Kiến trúc: DistilBertForTokenClassification
# Head: dropout(p=0.1) + classifier: Linear(768→3)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForTokenClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [9]:
from transformers import Trainer, TrainingArguments, DataCollatorForTokenClassification
import evaluate
import numpy as np

# Initialize evaluation metrics
metric = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    # Remove ignored index (if -100) and convert to labels
    true_labels = [[id2label[l] for l in label if l != -100] for label in labels]
    true_predictions = [
        [id2label[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    all_metrics = metric.compute(predictions=true_predictions, references=true_labels)
    return {
        "F1-score": all_metrics["overall_f1"],
        "precision": all_metrics["overall_precision"],
        "recall": all_metrics["overall_recall"],
        "accuracy": all_metrics["overall_accuracy"],
    }


training_args = TrainingArguments(
    output_dir="abte-restaurants-distilbert-base-uncased",
    logging_dir="logs",
    learning_rate=2e-5,
    per_device_train_batch_size=256,  # lớn hơn ATSC
    per_device_eval_batch_size=256,
    num_train_epochs=80,              # nhiều hơn ATSC (50→80)
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="F1-score",  # khác với ATSC (accuracy)
)

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=preprocessed_ds["train"],
    eval_dataset=preprocessed_ds["test"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)
trainer.train()

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Epoch,Training Loss,Validation Loss,F1-score,Precision,Recall,Accuracy
1,No log,0.461804,0.370053,0.435002,0.321979,0.817902
2,No log,0.287405,0.558023,0.466856,0.693436,0.872382
3,No log,0.238362,0.624795,0.590869,0.662854,0.906032
4,No log,0.203029,0.719373,0.720449,0.718299,0.924218
5,No log,0.183172,0.794554,0.784072,0.805321,0.937039
6,No log,0.177270,0.814328,0.811913,0.816758,0.938893
7,No log,0.192380,0.811344,0.804795,0.818001,0.937631
8,No log,0.185970,0.828410,0.820332,0.836648,0.942049
9,No log,0.193098,0.836029,0.824311,0.848086,0.942877
10,No log,0.215868,0.831887,0.829624,0.834162,0.941576


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=1200, training_loss=0.028163717687129976, metrics={'train_runtime': 3800.4561, 'train_samples_per_second': 75.822, 'train_steps_per_second': 0.316, 'total_flos': 6683942000369700.0, 'train_loss': 0.028163717687129976, 'epoch': 80.0})

In [10]:
from transformers import AutoTokenizer, AutoModelForTokenClassification, pipeline

tokenizer = AutoTokenizer.from_pretrained("distilbert/distilbert-base-uncased")

# Load best checkpoint
model = AutoModelForTokenClassification.from_pretrained(
    "/content/abte-restaurants-distilbert-base-uncased/checkpoint-1200"
)
id2label = {0: "O", 1: "B-Term", 2: "I-Term"}

# Pipeline tự động ghép B/I token thành entity
ate_pipe = pipeline(
    "token-classification",
    model=model,
    tokenizer=tokenizer,
    aggregation_strategy="simple"
)

def extract_aspects(text):
    predictions = ate_pipe(text)
    aspects = []
    for entity in predictions:
        if entity['entity_group'] == 'Term':
            aspects.append(entity['word'])
    return aspects

text = "The price was too high, but the cab was amazing."
aspects = extract_aspects(text)
print(f"Extracted aspects: {aspects}")
# Expected: ['price', 'cab']

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

Extracted aspects: ['price', 'cab']
